In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType

# Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Tool: Simple QA
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
qa_tool = Tool(
    name="Simple QA",
    func=qa_chain.run,
    description="Answer factual questions clearly"
)

C:\Users\my pc\AppData\Local\Temp\ipykernel_13376\2971102556.py:13: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
C:\Users\my pc\AppData\Local\Temp\ipykernel_13376\2971102556.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


In [2]:
# https://python.langchain.com/api_reference/langchain/memory.html?utm_source=chatgpt.com

In [4]:
#🧠 1. ConversationBufferMemory
from langchain.memory import ConversationBufferMemory

# Memory (stores chat history)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

#print(memory.chat_memory.messages)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

C:\Users\my pc\AppData\Local\Temp\ipykernel_17052\3944259546.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
C:\Users\my pc\AppData\Local\Temp\ipykernel_17052\3944259546.py:8: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initia

1️⃣ First Question


> Entering new AgentExecutor chain...
I should use Simple QA to get a clear answer to this question.
Action: Simple QA
Action Input: What is LangChain?
Observation: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.
Thought:I now know the final answer
Final Answer: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.

> Finished chain.

Answer: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.

2️⃣ Follow-up Question


> Entering new AgentExecutor chain...
I should use Simple QA to find out who 

# EVENTHOUGH MEMORY IS DECLARED BUT ZERO SHOT REACT IS INCAPABLE OF CARRYING OVER THE CONTEXT TO NEXT THREAD 

# IT HAS ONLY CAPABILITY TO STORE INFO

In [5]:
#🧠 1. ConversationBufferMemory
from langchain.memory import ConversationBufferMemory

# Memory (stores chat history)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,# ZERO_SHOT_REACT_DESCRIPTION
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

#print(memory.chat_memory.messages)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.
Thought:Do I need to use a tool? No
AI: LangChain is a blockchain-based platform that focuses on providing language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.

> Finished chain.

Answer: LangChain is a blockchain-based platform that focuses on providing language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.

2️⃣ Follow-up Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: Who create

In [6]:
#2)ConversationBufferWindowMemory

from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import initialize_agent, AgentType

memory = ConversationBufferWindowMemory(k=3,memory_key="chat_history")

agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

C:\Users\my pc\AppData\Local\Temp\ipykernel_17052\935981996.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=3,memory_key="chat_history")


1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.
Thought:Do I need to use a tool? No
AI: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.

> Finished chain.

Answer: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.

2️⃣ Follow-up Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: Who created LangChain?
Observa

In [7]:
#🧠 3. ConversationSummaryMemory
from langchain.memory import ConversationSummaryMemory
#from langchain.chat_models import ChatOpenAI

#llm = ChatOpenAI()

### hERE llm is required for creating the summary

memory = ConversationSummaryMemory(llm=llm,memory_key="chat_history")
# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)


# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

C:\Users\my pc\AppData\Local\Temp\ipykernel_17052\49606143.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm,memory_key="chat_history")


1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.
Thought:Do I need to use a tool? No
AI: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.

> Finished chain.

Answer: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.

2️⃣ Follow-up Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: Who 

In [10]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 16.4 MB/s eta 0:00:02
   ----- ---------------------------------- 2.6/18.9 MB 10.8 MB/s eta 0:00:02
   --------- ------------------------------ 4.5/18.9 MB 10.3 MB/s eta 0:00:02
   ------------- -------------------------- 6.6/18.9 MB 10.1 MB/s eta 0:00:02
   ------------------ --------------------- 8.7/18.9 MB 9.9 MB/s eta 0:00:02
   ---------------------- ----------------- 10.7/18.9 MB 10.0 MB/s eta 0:00:01
   --------------------------- ------------ 12.8/18.9 MB 9.9 MB/s eta 0:00:01
   ------------------------------- -------- 14.9/18.9 MB 9.9 MB/s eta 0:00:01
   ----------------------------------- ---- 16.8/18.9 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------  18.9/18.9 MB 9.8 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 9.0 MB/s  0:00:02


In [11]:
from langchain.memory import VectorStoreRetrieverMemory
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings()
retriever = FAISS.from_texts(["initial memory"], embedding).as_retriever()

memory = VectorStoreRetrieverMemory(retriever=retriever,memory_key="chat_history")
# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)


C:\Users\my pc\AppData\Local\Temp\ipykernel_17052\1771490952.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(retriever=retriever,memory_key="chat_history")


1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.
Thought:Do I need to use a tool? No
AI: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills.

> Finished chain.

Answer: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills.

2️⃣ Follow-up Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: Who created LangChain?
Observation: LangChain was created by a team of developers and language enthusiasts l

OpenAIEmbeddings(): Initializes the model that converts text into numerical vectors (embeddings). This allows the computer to "compare" the meaning of different sentences.

FAISS.from_texts(...): Creates a FAISS index (a fast local vector database). It starts with a dummy string "initial memory" because the index cannot be empty at creation.

.as_retriever(): Converts the vector store into a "retriever" object, which is a standardized interface LangChain uses to search for data.

VectorStoreRetrieverMemory: This is the "brain" of the operation. Unlike standard memory that stores a list of text, this memory saves every interaction into the FAISS database and searches that database for relevant context every time you ask a new question.

memory_key="chat_history": Tells the agent to look for a variable named "chat_history" in its prompt template to inject the retrieved memories.

agent.run(...): When these are called, two things happen behind the scenes:

The agent queries the VectorStore for past messages similar to the current question.

After the agent answers, it automatically saves the new Question-Answer pair back into the FAISS index for future use.

In [12]:
# Check what's stored in FAISS retriever
docs = retriever.vectorstore.similarity_search("memory")
for i, doc in enumerate(docs):
    print(f"🔹 Document {i+1}: {doc.page_content}")

🔹 Document 1: initial memory
🔹 Document 2: input: Explain it simply again.
output: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.
🔹 Document 3: input: What is LangChain?
output: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills.
🔹 Document 4: input: Who created it?
output: LangChain was created by a team of developers and language enthusiasts led by its founder, Dr. Alexey Malyshev.


similarity_search("memory"): Manually asks the FAISS database: "Show me all saved conversation snippets that are related to the word 'memory'."

docstore._dict.values(): This is a deep dive into the internal storage of FAISS to print every single thing currently stored in memory, regardless of relevance.

In [13]:
# See all texts in FAISS index
print("All indexed texts:")
for i, doc in enumerate(retriever.vectorstore.docstore._dict.values()):
    print(f"📄 {i+1}: {doc.page_content}")

All indexed texts:
📄 1: initial memory
📄 2: input: What is LangChain?
output: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills.
📄 3: input: Who created it?
output: LangChain was created by a team of developers and language enthusiasts led by its founder, Dr. Alexey Malyshev.
📄 4: input: Explain it simply again.
output: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.


In [16]:
query = "LangChain founder"
results = memory.retriever.get_relevant_documents(query)

print(f"\n🧠 Retrieval for: '{query}'")
for i, doc in enumerate(results):
    print(f"🔹 {i+1}: {doc.page_content}")



🧠 Retrieval for: 'LangChain founder'
🔹 1: input: Who is the founder of LangChain?
output: Harrison Chase is the founder of LangChain.
🔹 2: input: Who created it?
output: LangChain was created by a team of developers and language enthusiasts led by its founder, Dr. Alexey Malyshev.
🔹 3: input: What is LangChain?
output: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills.
🔹 4: input: Explain it simply again.
output: LangChain is a blockchain-based platform that focuses on language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.


In [15]:
# Manual storage of context in vector store

memory.save_context(
    {"input": "Who is the founder of LangChain?"},
    {"output": "Harrison Chase is the founder of LangChain."}
)


#🧠 6. PostgresChatMessageHistory, RedisChatMessageHistory, etc.

from langchain.memory import ConversationBufferMemory
from langchain.storage import PostgresChatMessageHistory

history = PostgresChatMessageHistory(session_id="abc123", connection_string="postgresql://...")
memory = ConversationBufferMemory(chat_memory=history)

# FilechatmessageHISTORY

In [4]:
from langchain.memory import FileChatMessageHistory

memory= FileChatMessageHistory(file_path = "ChatHistory.xlsx")

# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run({"Q":"What is LangChain?"})
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run({"Q":"Who created it?"})
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run({"Q":"Explain it simply again."})
print("\nAnswer:", res3)



ValidationError: 1 validation error for AgentExecutor
memory
  Input should be a valid dictionary or instance of BaseMemory [type=model_type, input_value=<langchain_community.chat...t at 0x000001BED68FCDF0>, input_type=FileChatMessageHistory]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

In [8]:
from langchain.memory import FileChatMessageHistory, ConversationBufferMemory

# 1. Define the storage (The "where")
# Note: LangChain usually uses .json or .txt for this. .xlsx isn't natively supported 
# by this class, so I recommend "chat_history.json".
message_history = FileChatMessageHistory(file_path="ChatHistory.txt")

# 2. Define the Memory Engine (The "how")
# We plug the 'history' into the 'chat_memory' parameter
memory = ConversationBufferMemory(memory_key="chat_history", chat_memory=message_history,return_messages=True)

# 3. Now the agent will accept it
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory  # Now 'memory' is a valid BaseMemory object
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: LangChain is a blockchain-based platform that aims to provide language learning services and opportunities for users to practice and improve their language skills through various interactive features and tools.
Thought:Do I need to use a tool? No
AI: LangChain is a blockchain-based platform that focuses on providing language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.

> Finished chain.

Answer: LangChain is a blockchain-based platform that focuses on providing language learning services and opportunities for users to practice and improve their language skills through interactive features and tools.

2️⃣ Follow-up Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: Who create